# 3. Setup + a Provider-Agnostic LLM Utility + Hello World Graph

By the end of the notebook you will have:

1. Working API keys for **Anthropic**, **OpenAI**, and **Google Gemini**.
2. A tiny, provider-agnostic utility module — `llm_utils.py` — with one function, `call_llm(provider, messages, system=None)`, that every later notebook in this course calls instead of ever touching a provider SDK directly from graph/node code.
3. Your first compiled, visualized, invoked LangGraph graph: a single node that calls `call_llm(...)`.
4. The first node of the **Applicant Screener** — the story we follow, with the same three candidates, through the next several videos.

**Why a utility function at all?** Once `call_llm` is the only way node code talks to a model, swapping the model behind the whole course becomes a one-line change (the `provider` argument) instead of a rewrite. This video is the *only* one that wires up and exercises all three providers side by side, specifically to prove that abstraction holds before the rest of the course commits to Claude as the default.

**Sections**
1. Setup & API keys
2. Raw sanity-check calls (one per provider, no utility yet)
3. Building `llm_utils.py`
4. Same prompt, three providers, side by side
5. Your first graph: `StateGraph`, `add_node`, `add_edge`, `START`, `END`
6. Visualizing the graph
7. The Applicant Screener story — building its first node

## 1. Setup & API keys

Grab one key from each of the three consoles below. All three have a free tier / trial credit, though some require a payment method on file:

| Provider | Where | Env var |
|---|---|---|
| Anthropic | console.anthropic.com → **API Keys** | `ANTHROPIC_API_KEY` |
| OpenAI | platform.openai.com → **API Keys** | `OPENAI_API_KEY` |
| Google Gemini | aistudio.google.com → **Get API Key** | `GOOGLE_API_KEY` |

Put them in a local `.env` file next to this notebook — **never hardcode a key in a cell, never commit `.env`**:

```
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_API_KEY=sk-...
GOOGLE_API_KEY=AIza...
```

We standardize on **current, non-legacy models** for every provider so the course doesn't teach against a model that's about to be deprecated:

- Claude → `claude-sonnet-5` (default provider/model for the rest of the course, from Video 4 onward)
- GPT → `gpt-5.1`
- Gemini → `gemini-3.5-flash`

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file if present

for env_var in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"]:
    if not os.environ.get(env_var):
        os.environ[env_var] = getpass.getpass(f"{env_var}: ")

print("All three keys are set:", all(os.environ.get(k) for k in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"]))

## 2. Raw sanity-check calls (one per provider, no utility yet)

Before building anything, confirm each key actually works by calling each SDK **directly**. This deliberately isolates three separate failure modes you might hit later — "is my API key bad?", "is my utility wrong?", "is my graph wrong?" — so if something breaks down the line you already know this layer is solid.

In [1]:
# --- Anthropic (Claude) ---
import anthropic

anthropic_client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
resp = anthropic_client.messages.create(
    model="claude-sonnet-5",
    max_tokens=64,
    messages=[{"role": "user", "content": "Say hi in exactly five words."}],
)
print("".join(block.text for block in resp.content if block.type == "text"))

Hi there! How are you?


In [ ]:
# --- OpenAI (GPT) ---
from openai import OpenAI

openai_client = OpenAI()  # reads OPENAI_API_KEY from env
resp = openai_client.chat.completions.create(
    model="gpt-5.1",
    messages=[{"role": "user", "content": "Say hi in exactly five words."}],
)
print(resp.choices[0].message.content)

In [2]:
# --- Google Gemini ---
from google import genai

gemini_client = genai.Client()  # reads GOOGLE_API_KEY from env
resp = gemini_client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Say hi in exactly five words.",
)
print(resp.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

All three keys work in isolation. Now let's stop calling three different SDKs with three different shapes, and build one function that hides the differences.

## 3. Building `llm_utils.py`

One function: `call_llm(provider, messages, system=None) -> LLMResponse`.

- `provider` is one of `"claude" | "gpt" | "gemini"`.
- `messages` is always the same plain shape regardless of provider: `[{"role": "user" | "assistant", "content": "..."}]`.
- `system` is a plain string system prompt, handled the right way for each provider under the hood (Anthropic and Gemini take it as a separate parameter; OpenAI's Chat Completions API wants it prepended as a `system` message).
- The return value, `LLMResponse`, always has a plain `.content` string, so calling code never needs to know which provider answered.

From here on, **every node in every later notebook calls `call_llm(...)` — never `anthropic.Anthropic()` / `OpenAI()` / `genai.Client()` directly.** Swapping the model behind the whole course becomes a one-line change to the `provider` argument (or a config/env value), never a rewrite of graph code.

In [ ]:
%%writefile llm_utils.py
"""Provider-agnostic LLM utility used by every node in this course.

Call `call_llm(provider, messages, system=None)` instead of ever importing a provider
SDK directly from graph/node code. Swapping models is then a one-line `provider` change.
"""
from dataclasses import dataclass

import anthropic
from openai import OpenAI
from google import genai
from google.genai import types as genai_types

# Standardize on one current, non-legacy model per provider.
CLAUDE_MODEL = "claude-sonnet-5"
GPT_MODEL = "gpt-5.1"
GEMINI_MODEL = "gemini-3.5-flash"

DEFAULT_PROVIDER = "claude"  # the rest of this course leaves this default in place

_anthropic_client = anthropic.Anthropic()
_openai_client = OpenAI()
_gemini_client = genai.Client()


@dataclass
class LLMResponse:
    content: str
    provider: str
    model: str
    raw: object  # the untouched provider response, for debugging


def _call_claude(messages: list[dict], system: str | None) -> LLMResponse:
    resp = _anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=system or anthropic.NOT_GIVEN,
        messages=messages,
    )
    # Newer Claude models may put a thinking block first, so join only the text blocks.
    text = "".join(block.text for block in resp.content if block.type == "text")
    return LLMResponse(content=text, provider="claude", model=CLAUDE_MODEL, raw=resp)


def _call_gpt(messages: list[dict], system: str | None) -> LLMResponse:
    full_messages = ([{"role": "system", "content": system}] if system else []) + messages
    resp = _openai_client.chat.completions.create(model=GPT_MODEL, messages=full_messages)
    return LLMResponse(content=resp.choices[0].message.content, provider="gpt", model=GPT_MODEL, raw=resp)


def _to_gemini_contents(messages: list[dict]) -> list[genai_types.Content]:
    # Gemini uses "model" instead of "assistant" for the assistant role.
    role_map = {"user": "user", "assistant": "model"}
    return [
        genai_types.Content(role=role_map[m["role"]], parts=[genai_types.Part(text=m["content"])])
        for m in messages
    ]


def _call_gemini(messages: list[dict], system: str | None) -> LLMResponse:
    config = genai_types.GenerateContentConfig(system_instruction=system) if system else None
    resp = _gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=_to_gemini_contents(messages),
        config=config,
    )
    return LLMResponse(content=resp.text, provider="gemini", model=GEMINI_MODEL, raw=resp)


_PROVIDERS = {"claude": _call_claude, "gpt": _call_gpt, "gemini": _call_gemini}


def call_llm(provider: str = DEFAULT_PROVIDER, messages: list[dict] | None = None, system: str | None = None) -> LLMResponse:
    if provider not in _PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}, expected one of {list(_PROVIDERS)}")
    return _PROVIDERS[provider](messages or [], system)

In [ ]:
from llm_utils import call_llm

resp = call_llm("claude", messages=[{"role": "user", "content": "Say hi in exactly five words."}])
print(resp)

From here on, this course calls `call_llm(...)` with the default provider (`"claude"`) left in place unless a cell says otherwise. If you want to point the whole course at GPT or Gemini instead, change `DEFAULT_PROVIDER` in `llm_utils.py` — nothing else needs to change.

## 4. Your first graph

The smallest possible real graph: one node, wired with `StateGraph` / `add_node` / `add_edge` / `START` / `END`, compiled and invoked. The state is a bare-bones `TypedDict` with one field in, one field out — just enough to be real.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class HelloState(TypedDict):
    input_text: str
    response_text: str


def hello_node(state: HelloState) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": state["input_text"]}])
    return {"response_text": result.content}


builder = StateGraph(HelloState)
builder.add_node("hello_node", hello_node)
builder.add_edge(START, "hello_node")
builder.add_edge("hello_node", END)

hello_graph = builder.compile()
hello_graph.invoke({"input_text": "What is LangGraph, in one sentence?"})

## 5. Visualizing the graph

Even a one-node graph is worth seeing rendered — this habit pays off once graphs get bigger.

In [ ]:
from IPython.display import Image, display


def show_graph(graph):
    """Render a compiled graph. Falls back to Mermaid text if the PNG renderer is unavailable."""
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_mermaid())


show_graph(hello_graph)

## 6. The story: Priya's Monday morning

Priya is the only recruiter at a small fintech company. One backend-engineer role is open, and the applications are piling up in her inbox as free-form text in every format imaginable. She can't read them all properly, and a keyword filter would reject good people on a technicality. So she wants an **Applicant Screener**: an assistant that reads each resume, judges it against her rubric, and tells her what to do with it.

Over the next few notebooks the Screener grows one upgrade at a time, and each upgrade fixes a complaint Priya has about the previous version. Three candidates apply, and we follow the same three through every notebook:

| Candidate | Resume | What we expect |
|---|---|---|
| **Hermione Granger** | Great: 7 years, Python, distributed systems, Kubernetes, SQL, leads a team | Strong match |
| **Harry Potter** | Mediocre: 4 years, Python/SQL/REST, no distributed systems or Kubernetes, informal mentoring | Borderline |
| **Ron Weasley** | Weak: under a year, HTML/CSS/JavaScript only, no leadership | Not a match |

**Today's problem:** the resumes are free text that Priya can't sort or compare.

**Today's fix:** one node that reads a resume and turns it into structured fields: name, years of experience, skills, and any leadership experience (her rubric will care about that soon). It only *extracts*. There is no judgment yet.

- **Input:** one raw resume string.
- **Output:** four structured fields: `candidate_name`, `years_experience`, `skills`, `leadership`.

In [ ]:
# The three candidates we follow through every notebook in this story.
RESUMES = {
    "Hermione": """
Hermione Granger
7 years of experience as a senior backend software engineer.
Skills: Python, distributed systems, SQL, PostgreSQL, Kubernetes, API design.
Led a 6-person platform team at a fintech company, mentoring junior engineers.
""",
    "Harry": """
Harry Potter
4 years of experience as a software developer.
Skills: Python, SQL, REST APIs.
Informally mentored two interns. No distributed systems or Kubernetes experience.
""",
    "Ron": """
Ron Weasley
8 months of experience as a junior web developer.
Skills: HTML, CSS, basic JavaScript.
No team or leadership experience.
""",
}

In [ ]:
class ScreenerState(TypedDict):
    resume_text: str
    candidate_name: str
    years_experience: str
    skills: str
    leadership: str


PARSE_SYSTEM_PROMPT = (
    "You extract structured fields from a resume. Reply with exactly four lines, "
    "no extra commentary:\n"
    "Name: <candidate name>\n"
    "Years: <years of experience as a number>\n"
    "Skills: <comma-separated skills>\n"
    "Leadership: <one short phrase on leadership or mentoring experience, or none>"
)


def parse_resume(state: ScreenerState) -> dict:
    result = call_llm(
        "claude",
        messages=[{"role": "user", "content": state["resume_text"]}],
        system=PARSE_SYSTEM_PROMPT,
    )

    fields = {"candidate_name": "", "years_experience": "", "skills": "", "leadership": ""}
    for line in result.content.strip().splitlines():
        if line.startswith("Name:"):
            fields["candidate_name"] = line.removeprefix("Name:").strip()
        elif line.startswith("Years:"):
            fields["years_experience"] = line.removeprefix("Years:").strip()
        elif line.startswith("Skills:"):
            fields["skills"] = line.removeprefix("Skills:").strip()
        elif line.startswith("Leadership:"):
            fields["leadership"] = line.removeprefix("Leadership:").strip()
    return fields


screener_builder = StateGraph(ScreenerState)
screener_builder.add_node("parse_resume", parse_resume)
screener_builder.add_edge(START, "parse_resume")
screener_builder.add_edge("parse_resume", END)

screener_graph = screener_builder.compile()
show_graph(screener_graph)
screener_graph.invoke({"resume_text": RESUMES["Hermione"]})

Hermione's resume went in as plain text and came out as four clean fields. The same single node should work for anyone, so let's run all three candidates through it:

In [ ]:
for name, resume in RESUMES.items():
    parsed = screener_graph.invoke({"resume_text": resume})
    print(f"{name}: {parsed['candidate_name']} | {parsed['years_experience']} yrs | {parsed['skills']} | leadership: {parsed['leadership']}")

Now Priya can see all three candidates side by side in the same shape. But a table of fields doesn't tell her whom to call, and she doesn't want fields, she wants a judgment.

**Next video** gives the Screener that judgment: more nodes, wired in a line, scoring each candidate against Priya's rubric.